# 使用scGPT进行单细胞数据整合分析

本教程演示如何使用scGPT模型进行单细胞数据整合分析。数据整合是单细胞数据分析中的重要任务，它可以将来自不同批次、不同条件或不同技术平台的数据整合到一起，消除批次效应并揭示细胞之间的真实生物学差异。


In [1]:
# 导入必要的库
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import matplotlib.pyplot as plt

# 添加scGPT路径
sys.path.insert(0, "../")
import scgpt as scg

print("✅ 所有库加载完成！")

In [2]:
# 设置参数
model_dir = Path("../save/scGPT_human")
data_dir = Path("../data/integration")
save_dir = Path("../results/integration")

# 创建保存目录
save_dir.mkdir(parents=True, exist_ok=True)

print("✅ 参数设置完成！")

In [3]:
# 加载多个批次的数据
batch1 = sc.read_h5ad(data_dir / "batch1.h5ad")
batch2 = sc.read_h5ad(data_dir / "batch2.h5ad")
batch3 = sc.read_h5ad(data_dir / "batch3.h5ad")

# 合并数据
adata = sc.concat(
    [batch1, batch2, batch3],
    keys=["batch1", "batch2", "batch3"],
    index_unique="-"
)

# 添加批次信息
adata.obs["batch"] = adata.obs["batch"].astype(str)

print(f"✅ 数据合并完成！总细胞数: {adata.n_obs}, 总基因数: {adata.n_vars}")
print(f"批次分布: {adata.obs['batch'].value_counts()}")

In [4]:
# 使用scGPT进行数据整合
integrated_adata = scg.tasks.integrate_data(
    adata,
    model_dir,
    batch_key="batch",
    batch_size=64,
    return_new_adata=True,
)

print("✅ 数据整合完成！")

In [5]:
# 可视化整合结果
# 1. 使用scGPT嵌入进行可视化
sc.pp.neighbors(integrated_adata, use_rep="X_scgpt")
sc.tl.umap(integrated_adata)

# 2. 绘制UMAP图，按批次着色
plt.figure(figsize=(12, 6))
sc.pl.umap(integrated_adata, color=["batch", "cell_type"], ncols=2, frameon=False)
plt.suptitle("scGPT整合结果可视化")
plt.tight_layout()
plt.show()

print("✅ 可视化完成！")

In [6]:
# 评估整合效果
metrics = scg.utils.eval_scib_metrics(
    integrated_adata,
    batch_key="batch",
    label_key="cell_type",
    embed_key="X_scgpt"
)

print("整合效果评估:")
for metric, value in metrics.items():
    print(f"  {metric}: {value:.4f}")

# 保存整合后的数据
integrated_adata.write_h5ad(save_dir / "integrated_data.h5ad")

print("✅ 整合分析完成！")